# Piloto/Análisis

análisis actualizado usando puntajes de `resultados_squad_es/scores`, sin reentrenar ni sobrescribirlos
- **Conglomerados por artículo** (`--cluster article`, leyendo `data/squad_es/dev.json` y `test.json`).
- **Semillas separadas** (`--seed-resample disjoint`, evitando que se elija para entrenamiento y evalación).
- **20,000 bootstrap** (`--boot 20000`, semilla `2026`).
- **Salida aislada** en `opcion_E_bootstrap/resultados_opcion_E/`.

> No requiere GPU: puede correrse directamente en entorno CPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/thesis'
%cd {BASE}/piloto_citca
!ls -la opcion_E_bootstrap

Mounted at /content/drive
/content/drive/MyDrive/thesis/piloto_citca
total 33
drwx------ 2 root root  4096 Nov 20 06:51 .
drwx------ 2 root root  4096 Oct 24 04:27 ..
-rw------- 1 root root 20124 Nov 20 06:53 analysis.py
-rw------- 1 root root  4635 Nov 20 07:20 correr_opcion_E.ipynb


## 1. Versiones del entorno (para reproducibilidad en el artículo)
Imprime las versiones exactas de Python y las librerías para completar los campos pendientes de la Sección de Metodología.

In [1]:
import sys, numpy, pandas, scipy, matplotlib
try:
    import torch, transformers
    ml_info = f' | torch {torch.__version__} | transformers {transformers.__version__}'
except ImportError:
    ml_info = ''

print(
    f'python {sys.version.split()[0]}'
    f' | numpy {numpy.__version__}'
    f' | pandas {pandas.__version__}'
    f' | scipy {scipy.__version__}'
    f' | matplotlib {matplotlib.__version__}'
    f'{ml_info}'
)

python 3.13.16 | numpy 2.4.6 | pandas 2.2.3 | scipy 1.16.3 | matplotlib 3.10.0 | torch 2.14.1+cu130 | transformers 5.22.1


## 2. Ejecutar análisis Opción E (20,000 réplicas)
Tarda alrededor de 1 minuto en CPU y guarda `summary.csv`, `config_means.csv`, `weighted_scores.csv`, `settings.json` y `fig_delta_cf.png` dentro de `opcion_E_bootstrap/resultados_opcion_E/`.

In [6]:
import os, subprocess, sys

ROOT = '/content/drive/MyDrive/thesis/piloto_citca'
SCRIPT = f'{ROOT}/opcion_E_bootstrap/analysis.py'
SCORES = f'{ROOT}/resultados_squad_es/scores'
DATA_DIR = f'{ROOT}/data/squad_es'
OUT_DIR = f'{ROOT}/opcion_E_bootstrap/resultados_opcion_E'

for label, p in [('SCRIPT', SCRIPT), ('SCORES', SCORES), ('DATA_DIR', DATA_DIR)]:
    print(f'{label}: {"OK" if os.path.exists(p) else "NO EXISTE"} -> {p}')

cmd = [
    sys.executable, SCRIPT,
    '--scores', SCORES,
    '--data-dir', DATA_DIR,
    '--cluster', 'article',
    '--seed-resample', 'disjoint',
    '--boot', '20000',
    '--seed', '2026',
    '--out', OUT_DIR,
]
res = subprocess.run(cmd, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print("ERROR:\n", res.stderr)

SCRIPT: NO EXISTE -> /content/drive/MyDrive/thesis/piloto_citca/opcion_E_bootstrap/analysis.py
SCORES: NO EXISTE -> /content/drive/MyDrive/thesis/piloto_citca/resultados_squad_es/scores
DATA_DIR: NO EXISTE -> /content/drive/MyDrive/thesis/piloto_citca/data/squad_es

ERROR:
 /usr/bin/python3: can't open file '/content/drive/MyDrive/thesis/piloto_citca/opcion_E_bootstrap/analysis.py': [Errno 2] No such file or directory



## 3. Revisar tablas y figura generadas

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary = pd.read_csv(f'{OUT_DIR}/summary.csv', dtype={'bits': str})
cols = ['model', 'quant', 'bits', 'n_clusters_dev', 'n_clusters_test',
        'delta_cf', 'boot_se', 'ci90_lo', 'ci90_hi', 'r_corr', 'decision']
print('////Tabla; artículos, semillas separadas, B=20,000 ')
display(summary[cols].round(2))

display(Image(f'{OUT_DIR}/fig_delta_cf.png'))

In [ ]:
weighted = pd.read_csv(f'{OUT_DIR}/weighted_scores.csv')
print('////Selección en validación')
dev_cols = ['model', 'config', 'fp32_dev_mean', 'gptq_8_dev_mean', 'rtn_8_dev_mean',
            'gptq_4_dev_mean', 'rtn_4_dev_mean', 'gptq_4g128_dev_mean', 'rtn_4g128_dev_mean',
            'P_lambda_dev']
display(weighted[dev_cols].round(2))

print('////ganadores')
winners = weighted.loc[weighted.groupby('model')['P_lambda_dev'].idxmax()]
test_cols = ['model', 'config', 'fp32_test_mean', 'gptq_8_test_mean', 'rtn_8_test_mean',
             'gptq_4_test_mean', 'rtn_4_test_mean', 'gptq_4g128_test_mean', 'rtn_4g128_test_mean',
             'P_lambda_test']
display(winners[test_cols].round(2))